# BloodMNIST validation-only staged Colab batch

Execute top to bottom on a Google Colab NVIDIA T4. Define multiple identified runs in the notebook, then execute them sequentially against one pinned commit. Each run calls modular repository APIs. A failed run is recorded and does not prevent later runs. The locked test split is never requested.


## 1. Runtime and source provenance

Fail closed on a non-T4 runtime and on an unset/unreviewed source commit.


In [ ]:
import re
import torch
assert torch.cuda.is_available(), "Select a Google Colab NVIDIA T4 runtime."
gpu_name = torch.cuda.get_device_name(0)
assert "T4" in gpu_name, f"T4 required; got {gpu_name}"
print("GPU:", gpu_name)


In [ ]:
REPO_URL = "https://github.com/james-chula/deep_learning_term_project.git"
COMMIT = "6b71dff8ea41fb599cc33497de88c56700900881"
assert len(COMMIT) == 40 and re.fullmatch(r"[0-9a-f]{40}", COMMIT), "Set COMMIT to the Lead-approved full SHA."
%cd /content
!rm -rf /content/project
!git clone -q $REPO_URL /content/project
%cd /content/project
!git checkout --detach $COMMIT
!pip install -q -r requirements-colab.txt
actual_commit = !git rev-parse HEAD
assert actual_commit[0] == COMMIT
print("Pinned commit:", actual_commit[0])
# The unchanged command-line entry point remains: python run_experiment.py


## 2. Batch configuration

Each entry has a unique notebook-only `id` and a `config` override dictionary. Repository defaults are copied fresh for every run, so overrides cannot leak between runs. Unknown keys, duplicate IDs, invalid modes, and duplicate run names fail before execution.


In [ ]:
from pathlib import Path
from copy import deepcopy
import importlib
import os
import sys

os.environ["MEDMNIST_DATA_ROOT"] = "/content/medmnist"
PROJECT_ROOT = Path("/content/project").resolve()
CONFIG_FILE = PROJECT_ROOT / "config.py"
if not CONFIG_FILE.is_file():
    raise FileNotFoundError(f"Missing configuration file: {CONFIG_FILE}")

project_path = str(PROJECT_ROOT)
src_path = str(PROJECT_ROOT / "src")
sys.path = [p for p in sys.path if p not in {project_path, src_path}]
sys.path.insert(0, src_path)
sys.path.insert(0, project_path)
sys.path_importer_cache.pop(project_path, None)
sys.path_importer_cache.pop(src_path, None)
importlib.invalidate_caches()
sys.modules.pop("config", None)

from config import CONFIG
BASE_CONFIG = deepcopy(CONFIG)
RUN_CONFIGS = [
    {
        "id": "majority-1pct",
        "config": {
            "mode": "majority",
            "run_name": "bloodmnist_majority_1pct_seed2026",
            "use_fixed_1pct_subset": True,
            "seed": 2026,
        },
    },
    {
        "id": "linear-probe-1pct",
        "config": {
            "mode": "linear_probe",
            "run_name": "bloodmnist_linear_probe_1pct_seed2026",
            "use_fixed_1pct_subset": True,
            "seed": 2026,
            "epochs": 10,
            "batch_size": 64,
            "learning_rate": 0.001,
        },
    },
    {
        "id": "finetune-1pct",
        "config": {
            "mode": "finetune",
            "run_name": "bloodmnist_finetune_1pct_seed2026",
            "use_fixed_1pct_subset": True,
            "seed": 2026,
            "epochs": 10,
            "batch_size": 64,
            "learning_rate": 0.001,
        },
    },
]

def validate_run_configs(run_configs):
    allowed_modes = {"majority", "linear_probe", "finetune"}
    known_keys = set(BASE_CONFIG)
    ids, names = set(), set()
    for index, spec in enumerate(run_configs):
        if set(spec) != {"id", "config"}:
            raise ValueError(f"RUN_CONFIGS[{index}] must contain exactly 'id' and 'config'.")
        run_id, overrides = spec["id"], spec["config"]
        if not isinstance(run_id, str) or not run_id.strip():
            raise ValueError(f"RUN_CONFIGS[{index}].id must be a non-empty string.")
        if run_id in ids:
            raise ValueError(f"Duplicate run id: {run_id}")
        if not isinstance(overrides, dict):
            raise TypeError(f"Config for {run_id} must be a dictionary.")
        unknown = set(overrides) - known_keys
        if unknown:
            raise ValueError(f"Unknown config keys for {run_id}: {sorted(unknown)}")
        merged = {**BASE_CONFIG, **overrides}
        if merged["mode"] not in allowed_modes:
            raise ValueError(f"Invalid mode for {run_id}: {merged['mode']}")
        if not isinstance(merged["run_name"], str) or not merged["run_name"].strip():
            raise ValueError(f"run_name for {run_id} must be a non-empty string.")
        if merged["run_name"] in names:
            raise ValueError(f"Duplicate run_name: {merged['run_name']}")
        ids.add(run_id)
        names.add(merged["run_name"])

validate_run_configs(RUN_CONFIGS)
RUN_CONFIGS


## 3. Reusable notebook orchestration

The functions below repeat data setup, optional sample/model preview, training, selected-checkpoint evaluation, diagnostics, and artifact packaging for one merged configuration. Model and experiment logic remains in modular `.py` files.


In [ ]:
import json
import time
import torch
import matplotlib.pyplot as plt
from IPython.display import Image, display

from data import get_data_loaders
from model import create_model, majority_probabilities
from train import validate_majority
from run_experiment import (
    class_names_from_audit, create_artifact_zip, current_commit,
    display_class_name, evaluate_best_checkpoint, make_output_folder,
    package_versions, require_t4, save_json, save_validation_diagnostics,
    set_seed, train_neural_mode, validation_metrics,
)

def apply_run_config(overrides):
    CONFIG.clear()
    CONFIG.update(deepcopy(BASE_CONFIG))
    CONFIG.update(deepcopy(overrides))
    return CONFIG

def show_data_preview(train_loader, class_names):
    images, labels = next(iter(train_loader))
    mean = torch.tensor((0.485, 0.456, 0.406)).view(3, 1, 1)
    std = torch.tensor((0.229, 0.224, 0.225)).view(3, 1, 1)
    figure, axes = plt.subplots(2, 4, figsize=(12, 6))
    for axis, image, label in zip(axes.flat, images[:8], labels.view(-1)[:8]):
        axis.imshow((image.cpu() * std + mean).clamp(0, 1).permute(1, 2, 0))
        axis.set_title(display_class_name(label, class_names))
        axis.axis("off")
    figure.suptitle(f"Audited training samples: {CONFIG['run_name']}")
    plt.tight_layout()
    plt.show()

def show_model_preview():
    if CONFIG["mode"] == "majority":
        print("Majority baseline: no neural model or optimizer.")
        return
    model_preview = create_model(CONFIG["num_classes"], CONFIG["mode"])
    total = sum(parameter.numel() for parameter in model_preview.parameters())
    trainable = sum(parameter.numel() for parameter in model_preview.parameters() if parameter.requires_grad)
    print({"total_parameters": total, "trainable_parameters": trainable})
    del model_preview

def show_saved_figures(output_folder):
    for filename in [
        "learning_curves.png", "confusion_matrix_counts.png",
        "confusion_matrix_normalized.png", "per_class_metrics.png",
        "misclassified_samples.png",
    ]:
        path = output_folder / filename
        if path.is_file():
            print(filename)
            display(Image(filename=str(path)))


## 4. One-run function

`run_one` receives an ID plus overrides, applies a fresh configuration, and returns a compact success record. Exceptions are handled by the outer batch loop so later configurations still run.


In [ ]:
def run_one(run_id, overrides, *, show_previews=True, show_figures=True):
    apply_run_config(overrides)
    print(f"\n===== RUN {run_id}: {CONFIG['run_name']} ({CONFIG['mode']}) =====")
    assert current_commit() == COMMIT
    set_seed(CONFIG["seed"])
    started = time.time()

    train_loader, validation_loader, provider_metadata = get_data_loaders(CONFIG)
    assert provider_metadata["test_policy"] == "locked_not_loaded"
    class_names = class_names_from_audit(CONFIG["audit_manifest_path"])
    print("loader counts:", len(train_loader.dataset), len(validation_loader.dataset))
    if show_previews:
        show_data_preview(train_loader, class_names)
        show_model_preview()

    # Reset after preview iteration/model creation for reproducible training.
    set_seed(CONFIG["seed"])
    output_folder = make_output_folder()
    run_metadata = {
        "run_id": run_id,
        "config": dict(CONFIG),
        "provider": provider_metadata,
        "git_commit": current_commit(),
        "gpu": require_t4(),
        "package_versions": package_versions(),
        "test_evaluated": False,
        "test_policy": "locked_not_loaded",
    }
    save_json(output_folder / "run_metadata.json", run_metadata)
    save_json(output_folder / "config.json", CONFIG)

    device = torch.device("cuda")
    if CONFIG["mode"] == "majority":
        train_labels = []
        for _, batch_labels in train_loader:
            train_labels.extend(batch_labels.view(-1).tolist())
        probabilities, majority_class = majority_probabilities(
            train_labels, len(validation_loader.dataset), CONFIG["num_classes"]
        )
        selected_result = validate_majority(probabilities, validation_loader)
        selected_result["probabilities"] = probabilities.numpy()
        best_epoch, history = None, []
    else:
        best_epoch, best_f1, _, history = train_neural_mode(
            CONFIG["mode"], train_loader, validation_loader, device, output_folder
        )
        _, checkpoint_epoch, selected_result = evaluate_best_checkpoint(
            CONFIG["mode"], validation_loader, device, output_folder
        )
        assert checkpoint_epoch == best_epoch

    metrics = validation_metrics(
        selected_result["labels"], selected_result["predictions"],
        selected_result.get("probabilities"),
    )
    summary = {
        "run_id": run_id,
        "run_name": CONFIG["run_name"],
        "mode": CONFIG["mode"],
        "best_epoch": best_epoch,
        "best_validation_macro_f1": metrics["macro_f1"],
        "runtime_seconds": time.time() - started,
        "test_evaluated": False,
    }
    if CONFIG["mode"] == "majority":
        summary["majority_class"] = majority_class
    save_json(output_folder / "metrics.json", metrics)
    save_json(output_folder / "summary.json", summary)
    save_validation_diagnostics(
        selected_result["labels"], selected_result["predictions"], metrics,
        validation_loader, class_names, output_folder,
    )
    archive_path = create_artifact_zip(output_folder)
    if show_figures:
        show_saved_figures(output_folder)
    print(json.dumps(summary, indent=2))
    print("Artifact ZIP:", archive_path)
    return {
        "id": run_id, "status": "succeeded", "summary": summary,
        "metrics": metrics, "output_folder": str(output_folder),
        "archive_path": str(archive_path),
    }


## 5. Execute the batch

Each failure is caught, printed with its run ID, and retained in `BATCH_RESULTS`. Execution then continues with the next configuration.


In [ ]:
import traceback

BATCH_RESULTS = []
for spec in RUN_CONFIGS:
    run_id = spec["id"]
    try:
        result = run_one(run_id, spec["config"])
    except Exception as error:
        result = {
            "id": run_id,
            "status": "failed",
            "error_type": type(error).__name__,
            "error": str(error),
            "traceback": traceback.format_exc(),
        }
        print(f"FAILED [{run_id}] {type(error).__name__}: {error}")
        traceback.print_exc()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    BATCH_RESULTS.append(result)


## 6. Batch status and download

The final table identifies every succeeded or failed run. A batch manifest is saved separately; successful run ZIPs remain independent evidence packages. Download all generated files in one batch ZIP.


In [ ]:
import shutil
from datetime import datetime, timezone

status_rows = []
for result in BATCH_RESULTS:
    row = {"id": result["id"], "status": result["status"]}
    if result["status"] == "succeeded":
        row.update({
            "mode": result["summary"]["mode"],
            "macro_f1": result["metrics"]["macro_f1"],
            "accuracy": result["metrics"]["accuracy"],
            "artifact": result["archive_path"],
        })
    else:
        row.update({"error_type": result["error_type"], "error": result["error"]})
    status_rows.append(row)

print(json.dumps(status_rows, indent=2))
failed_ids = [row["id"] for row in status_rows if row["status"] == "failed"]
print("Failed run IDs:", failed_ids if failed_ids else "none")

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
batch_dir = Path(CONFIG["output_root"]) / f"{stamp}_batch_manifest"
batch_dir.mkdir(parents=True, exist_ok=False)
save_json(batch_dir / "batch_results.json", BATCH_RESULTS)
for result in BATCH_RESULTS:
    if result["status"] == "succeeded":
        shutil.copy2(result["archive_path"], batch_dir / Path(result["archive_path"]).name)
batch_archive = shutil.make_archive(str(batch_dir), "zip", root_dir=batch_dir.parent, base_dir=batch_dir.name)
print("Batch ZIP:", batch_archive)

from google.colab import files
files.download(batch_archive)
